In [1]:
# ============================================================
# Seeded train+eval for the "multiple seeds" ablation (Tier 1 / #3).
# Loops over SEEDS automatically for whichever ARCH/CATEGORY you set below --
# change ARCH/CATEGORY once per config, not once per seed.
#
# Matches the two reference implementations exactly:
#   - Vial architecture  <- Tiled_DINOv2_Code.docx   (teacher layers 6+11 concat,
#     bottleneck TransformerStudent that takes the TEACHER'S OWN TOKENS as input,
#     mixed whole-image + dense-tile training set, Adam betas=(0.5,0.999), no LR
#     scheduler, pure cosine loss, 150 epochs for both home and cross)
#   - Fabric architecture <- fabric-og.ipynb         (single-layer teacher, raw-pixel
#     TransformerStudent, one random tile per image per batch shuffled every epoch,
#     Adam default betas, CosineAnnealingLR, 0.5*cosine + 0.5*MSE loss, 50 epochs)
#
# Covers all four matrix cells by changing ARCH / CATEGORY at the top:
#   1. vial-home     : ARCH="vial",   CATEGORY="vial"     -- seed 42 already done
#   2. fabric-home    : ARCH="fabric", CATEGORY="fabric"
#   3. vial-on-fabric : ARCH="vial",   CATEGORY="fabric"
#   4. fabric-on-vial : ARCH="fabric", CATEGORY="vial"
# Each config run loops SEEDS below and appends one row per seed to
# /kaggle/working/seed_results.csv, saving one checkpoint per seed.
# ============================================================
import os, glob, random, csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy import ndimage
from sklearn.metrics import roc_auc_score

# ---------------- CHANGE THESE PER CONFIG ----------------
ARCH = "vial"          # "vial" or "fabric"
CATEGORY = "fabric"      # "vial" or "fabric"
SEEDS = [42, 43, 44]    # seed 42 already done for vial/vial -- reset to [42, 43, 44] for a fresh config
# -----------------------------------------------------------

EPOCHS_VIAL_CROSS = 150   # vial-arch trained on fabric data: same budget as vial-home (confirmed OK, no crashing)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_ROOT = "/kaggle/input/datasets/arhaan1234/mv-tech-ad2/mvtec_ad_2"
DATA_ROOT = os.path.join(DATASET_ROOT, CATEGORY)
RESULTS_CSV = "/kaggle/working/seed_results.csv"

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def find_images(root):
    exts = ("*.png", "*.jpg", "*.jpeg", "*.PNG", "*.JPG", "*.JPEG")
    paths = []
    for ext in exts:
        paths += glob.glob(os.path.join(root, "**", ext), recursive=True)
    return sorted(set(paths))

train_paths = find_images(os.path.join(DATA_ROOT, "train", "good"))
test_paths_all = find_images(os.path.join(DATA_ROOT, "test_public"))
test_paths = [p for p in test_paths_all if "ground_truth" not in p]
print(f"[{ARCH} on {CATEGORY}] train={len(train_paths)} test={len(test_paths)}  seeds={SEEDS}")

# ============================================================
# Image loading (seed-independent -- defined once)
# ============================================================
def load_square(path, size, is_mask=False):
    im = Image.open(path).convert("L" if is_mask else "RGB").resize((size, size), Image.NEAREST if is_mask else Image.BILINEAR)
    return (np.array(im) > 127).astype(np.uint8) if is_mask else torch.from_numpy(np.array(im)).float().permute(2, 0, 1) / 255

def load_resize_crop(path, resize_to, crop_to, is_mask=False):
    im = Image.open(path).convert("L" if is_mask else "RGB")
    w, h = im.size; s = resize_to / min(w, h)
    im = im.resize((max(round(w * s), resize_to), max(round(h * s), resize_to)), Image.NEAREST if is_mask else Image.BILINEAR)
    w, h = im.size; l, t = (w - crop_to) // 2, (h - crop_to) // 2
    im = im.crop((l, t, l + crop_to, t + crop_to))
    return (np.array(im) > 127).astype(np.uint8) if is_mask else torch.from_numpy(np.array(im)).float().permute(2, 0, 1) / 255

def imagenet_normalize(img):
    return (img - IMAGENET_MEAN.to(img.device)) / IMAGENET_STD.to(img.device)

def _load_eval(path, size, is_mask=False):
    return load_resize_crop(path, size, size, is_mask) if ARCH == "vial" else load_square(path, size, is_mask)

# ============================================================
# Teacher (seed-independent, frozen, loaded once)
# ============================================================
dinov2 = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
dinov2.eval()
for p in dinov2.parameters(): p.requires_grad = False
dinov2 = dinov2.to(DEVICE)

@torch.no_grad()
def teacher_out(batch):
    if ARCH == "vial":
        return torch.cat(dinov2.get_intermediate_layers(batch, n=(6, 11), reshape=False), -1)
    return dinov2.forward_features(batch)["x_norm_patchtokens"]

# ============================================================
# ARCH-specific config + precomputed data (seed-independent -- built ONCE, reused every seed)
# ============================================================
if ARCH == "vial":
    TILE_SIZE, TILE_STRIDE, SRC_SIZE = 224, 112, 448
    BATCH_SIZE, BETAS, LR = 16, (0.5, 0.999), 1e-3
    EPOCHS = 150 if CATEGORY == "vial" else EPOCHS_VIAL_CROSS
    TEACHER_DIM = 1536
    NUM_TOKENS = (TILE_SIZE // 14) ** 2
    BOTTLENECK_DIM, STUDENT_DEPTH, STUDENT_HEADS = 128, 3, 4

    class Student(nn.Module):
        def __init__(self):
            super().__init__()
            self.down_proj = nn.Linear(TEACHER_DIM, BOTTLENECK_DIM)
            self.pos_embed = nn.Parameter(torch.zeros(1, NUM_TOKENS, BOTTLENECK_DIM))
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
            self.blocks = nn.ModuleList([nn.TransformerEncoderLayer(
                d_model=BOTTLENECK_DIM, nhead=STUDENT_HEADS, dim_feedforward=int(BOTTLENECK_DIM * 4),
                dropout=0.0, activation="gelu", batch_first=True, norm_first=True) for _ in range(STUDENT_DEPTH)])
            self.norm = nn.LayerNorm(BOTTLENECK_DIM)
            self.up_proj = nn.Linear(BOTTLENECK_DIM, TEACHER_DIM)
        def forward(self, tokens):
            x = self.down_proj(tokens) + self.pos_embed
            for b in self.blocks: x = b(x)
            return self.up_proj(self.norm(x))

    def feature_loss(s_out, t_out):
        return (1 - F.cosine_similarity(s_out, t_out, dim=-1)).mean()

    print("Pre-computing the mixed whole-image + dense-tile training set (once, reused across seeds)...")
    def _grid_coords(n, tile, stride):
        xs = list(range(0, max(n - tile, 0) + 1, stride))
        if xs[-1] != n - tile: xs.append(n - tile)
        return xs
    TILE_COORDS = [(y, x) for y in _grid_coords(SRC_SIZE, TILE_SIZE, TILE_STRIDE)
                           for x in _grid_coords(SRC_SIZE, TILE_SIZE, TILE_STRIDE)]
    whole_tensors = [load_resize_crop(p, 256, 224) for p in train_paths]
    source_tensors = [load_resize_crop(p, SRC_SIZE, SRC_SIZE) for p in train_paths]
    tile_entries = [(img_idx, y, x) for img_idx in range(len(train_paths)) for (y, x) in TILE_COORDS]
    combined_len = len(whole_tensors) + len(tile_entries)
    print(f"Whole images: {len(whole_tensors)}, Tiles: {len(tile_entries)}, Combined: {combined_len}")

    def get_combined_item(idx):
        if idx < len(whole_tensors):
            return whole_tensors[idx]
        img_idx, y, x = tile_entries[idx - len(whole_tensors)]
        return source_tensors[img_idx][:, y:y + TILE_SIZE, x:x + TILE_SIZE]

    def new_student(): return Student().to(DEVICE)

    def train_one(student, seed):
        optimizer = torch.optim.Adam(student.parameters(), lr=LR, betas=BETAS)
        student.train()
        order = list(range(combined_len))
        for epoch in range(EPOCHS):
            random.shuffle(order)
            epoch_loss, n_batches = 0.0, 0
            for i in range(0, combined_len, BATCH_SIZE):
                idxs = order[i:i + BATCH_SIZE]
                tiles = torch.stack([get_combined_item(j) for j in idxs]).to(DEVICE)
                tiles_norm = imagenet_normalize(tiles)
                with torch.no_grad():
                    t_out = teacher_out(tiles_norm)
                s_out = student(t_out)
                loss = feature_loss(s_out, t_out)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                epoch_loss += loss.item(); n_batches += 1
            if epoch % 15 == 0 or epoch == EPOCHS - 1:
                print(f"  [seed {seed}] epoch {epoch:3d}  loss {epoch_loss / max(n_batches,1):.5f}")

else:  # fabric
    TILE_SIZE, TILE_STRIDE, SRC_SIZE = 518, 392, 1036
    BATCH_SIZE, BETAS, LR = 8, (0.9, 0.999), 1e-3
    EPOCHS = 50
    PATCH_SIZE = 14
    STUDENT_EMBED_DIM, STUDENT_DEPTH, STUDENT_HEADS = 384, 4, 6

    class Student(nn.Module):
        def __init__(self):
            super().__init__()
            self.patch_embed = nn.Conv2d(3, STUDENT_EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
            n_patches = (TILE_SIZE // PATCH_SIZE) ** 2
            self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, STUDENT_EMBED_DIM))
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
            layer = nn.TransformerEncoderLayer(d_model=STUDENT_EMBED_DIM, nhead=STUDENT_HEADS,
                                                dim_feedforward=STUDENT_EMBED_DIM * 4, batch_first=True)
            self.blocks = nn.TransformerEncoder(layer, num_layers=STUDENT_DEPTH)
            self.head = nn.Linear(STUDENT_EMBED_DIM, 768)
        def forward(self, x):
            x = self.patch_embed(x).flatten(2).transpose(1, 2) + self.pos_embed
            return self.head(self.blocks(x))

    def feature_loss(s_out, t_out, alpha=0.5):
        cos_loss = (1 - F.cosine_similarity(s_out, t_out, dim=-1)).mean()
        return alpha * cos_loss + (1 - alpha) * F.mse_loss(s_out, t_out)

    def _coords(n, tile, stride):
        xs = list(range(0, n - tile + 1, stride))
        return xs + [n - tile] if xs[-1] != n - tile else xs
    TILE_COORDS = [(y, x) for y in _coords(SRC_SIZE, TILE_SIZE, TILE_STRIDE) for x in _coords(SRC_SIZE, TILE_SIZE, TILE_STRIDE)]

    def random_tile_from(path):
        img = load_square(path, SRC_SIZE)
        y, x = random.choice(TILE_COORDS)
        return img[:, y:y + TILE_SIZE, x:x + TILE_SIZE]

    def new_student(): return Student().to(DEVICE)

    def train_one(student, seed):
        optimizer = torch.optim.Adam(student.parameters(), lr=LR, betas=BETAS)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        student.train()
        paths = train_paths.copy()
        for epoch in range(EPOCHS):
            random.shuffle(paths)
            epoch_loss, n_batches = 0.0, 0
            for i in range(0, len(paths), BATCH_SIZE):
                batch_paths = paths[i:i + BATCH_SIZE]
                tiles = torch.stack([random_tile_from(p) for p in batch_paths]).to(DEVICE)
                tiles_norm = imagenet_normalize(tiles)
                with torch.no_grad():
                    t_out = teacher_out(tiles_norm)
                s_out = student(tiles_norm)
                loss = feature_loss(s_out, t_out)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                epoch_loss += loss.item(); n_batches += 1
            scheduler.step()
            if epoch % 10 == 0 or epoch == EPOCHS - 1:
                print(f"  [seed {seed}] epoch {epoch:3d}  loss {epoch_loss / max(n_batches,1):.4f}")

# ============================================================
# Shared scoring harness (seed-independent)
# ============================================================
_trapz = getattr(np, "trapezoid", None) or np.trapz

def shrink(maps, masks, size=448):
    if maps[0].shape[0] == size:
        return maps, [(g > 0).astype(np.uint8) for g in masks]
    m2 = [F.interpolate(torch.from_numpy(np.asarray(m, np.float32))[None, None], size=(size, size),
                        mode="bilinear", align_corners=False, antialias=True)[0, 0].numpy() for m in maps]
    g2 = [F.interpolate(torch.from_numpy((np.asarray(g) > 0).astype(np.float32))[None, None], size=(size, size),
                        mode="nearest")[0, 0].numpy().astype(np.uint8) for g in masks]
    return m2, g2

def _aupro(maps, gts, thresholds, fpr_limit):
    regions = []
    for i, g in enumerate(gts):
        lab, n = ndimage.label(g)
        regions += [(i, lab == r, int((lab == r).sum())) for r in range(1, n + 1)]
    total_normal = sum(int((g == 0).sum()) for g in gts)
    fprs, pros = [], []
    for t in thresholds:
        preds = [m >= t for m in maps]
        fp = sum(int(np.logical_and(p, g == 0).sum()) for p, g in zip(preds, gts))
        pros.append(np.mean([np.logical_and(preds[i], rm).sum() / sz for i, rm, sz in regions]))
        fprs.append(fp / max(total_normal, 1))
    fprs, pros = np.array(fprs), np.array(pros)
    keep = fprs <= fpr_limit
    if keep.sum() < 2: return 0.0
    o = np.argsort(fprs[keep])
    return float(_trapz(pros[keep][o], fprs[keep][o]) / fpr_limit)

def aupro_quantile(maps, masks, n=1000, fpr_limit=0.05):
    gts = [(np.asarray(g) > 0).astype(np.uint8) for g in masks]
    normal = np.concatenate([m[g == 0].ravel() for m, g in zip(maps, gts)])
    thr = np.unique(np.quantile(normal, np.linspace(0, 1, n)))[::-1]
    return _aupro(maps, gts, thr, fpr_limit)

def report(maps, masks, labels):
    maps, masks = shrink(maps, masks)
    y = np.array(labels)
    def topk(m, margin=10, k=50):
        c = m[margin:-margin, margin:-margin].ravel()
        return float(np.partition(c, -k)[-k:].mean())
    i_auroc = roc_auc_score(y, [topk(m) for m in maps])
    p_auroc = roc_auc_score(np.concatenate([g[::4, ::4].ravel() for g in masks]),
                            np.concatenate([m[::4, ::4].ravel() for m in maps]))
    aupro = aupro_quantile(maps, masks, n=1000)
    return i_auroc, p_auroc, aupro

def evaluate(student):
    student.eval()
    def _compute_map_for(path):
        x = (_load_eval(path, SRC_SIZE) - IMAGENET_MEAN) / IMAGENET_STD
        acc = np.zeros((SRC_SIZE, SRC_SIZE), np.float32); cnt = np.zeros_like(acc)
        for i in range(0, len(TILE_COORDS), 8):
            cs = TILE_COORDS[i:i + 8]
            b = torch.stack([x[:, y:y + TILE_SIZE, xx:xx + TILE_SIZE] for y, xx in cs]).to(DEVICE)
            with torch.no_grad():
                t_out = teacher_out(b)
                s_out = student(t_out) if ARCH == "vial" else student(b)
                err = 1 - F.cosine_similarity(s_out, t_out, dim=-1)
            g = int(err.shape[1] ** 0.5)
            err = F.interpolate(err.view(-1, 1, g, g), size=(TILE_SIZE, TILE_SIZE), mode="bilinear", align_corners=False)[:, 0].cpu().numpy()
            for (y, xx), m in zip(cs, err):
                acc[y:y + TILE_SIZE, xx:xx + TILE_SIZE] += m; cnt[y:y + TILE_SIZE, xx:xx + TILE_SIZE] += 1
        return acc / cnt

    root = f"{DATA_ROOT}/test_public"
    maps, masks, labels = [], [], []
    for sub in ("good", "bad"):
        for f in sorted(os.listdir(f"{root}/{sub}")):
            if not f.lower().endswith((".png", ".jpg", ".jpeg")): continue
            maps.append(_compute_map_for(f"{root}/{sub}/{f}")); labels.append(int(sub == "bad"))
            stem = os.path.splitext(f)[0]
            mp = next((c for c in (f"{root}/ground_truth/bad/{stem}_mask.png", f"{root}/ground_truth/bad/{stem}.png") if os.path.exists(c)), None) if sub == "bad" else None
            masks.append(_load_eval(mp, 448, True) if mp else np.zeros((448, 448), np.uint8))
    return report(maps, masks, labels)

# ============================================================
# Main loop over seeds
# ============================================================
file_exists = os.path.exists(RESULTS_CSV)
for seed in SEEDS:
    print(f"\n{'='*60}\n[{ARCH} on {CATEGORY}] SEED {seed}\n{'='*60}")
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    student = new_student()
    train_one(student, seed)

    ckpt_path = f"/kaggle/working/seed_ckpt_{ARCH}_{CATEGORY}_seed{seed}.pt"
    torch.save({"student_state_dict": student.state_dict(), "arch": ARCH, "category": CATEGORY,
                "seed": seed, "epochs": EPOCHS}, ckpt_path)
    print(f"saved checkpoint: {ckpt_path}")

    i_auroc, p_auroc, aupro = evaluate(student)
    print(f"[{ARCH} design on {CATEGORY}, seed {seed}]  I-AUROC={i_auroc:.4f} | P-AUROC={p_auroc:.4f} | AU-PRO@5%={aupro:.4f}")

    row = dict(arch=ARCH, category=CATEGORY, seed=seed, epochs=EPOCHS,
               i_auroc=round(i_auroc, 4), p_auroc=round(p_auroc, 4), aupro=round(aupro, 4))
    with open(RESULTS_CSV, "a", newline="") as fp:
        w = csv.DictWriter(fp, fieldnames=list(row.keys()))
        if not file_exists: w.writeheader(); file_exists = True
        w.writerow(row)
    print(f"appended to {RESULTS_CSV}")

    del student
    torch.cuda.empty_cache()

print("\nAll seeds done for this config.")

[vial on fabric] train=387 test=156  seeds=[42, 43, 44]
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:01<00:00, 241MB/s]


Pre-computing the mixed whole-image + dense-tile training set (once, reused across seeds)...
Whole images: 387, Tiles: 3483, Combined: 3870

[vial on fabric] SEED 42
  [seed 42] epoch   0  loss 0.13573
  [seed 42] epoch  15  loss 0.02660
  [seed 42] epoch  30  loss 0.02630
  [seed 42] epoch  45  loss 0.02614
  [seed 42] epoch  60  loss 0.02611
  [seed 42] epoch  75  loss 0.02602
  [seed 42] epoch  90  loss 0.02598
  [seed 42] epoch 105  loss 0.02607
  [seed 42] epoch 120  loss 0.02594
  [seed 42] epoch 135  loss 0.02592
  [seed 42] epoch 149  loss 0.02591
saved checkpoint: /kaggle/working/seed_ckpt_vial_fabric_seed42.pt
[vial design on fabric, seed 42]  I-AUROC=0.6333 | P-AUROC=0.9080 | AU-PRO@5%=0.3545
appended to /kaggle/working/seed_results.csv

[vial on fabric] SEED 43
  [seed 43] epoch   0  loss 0.13560
  [seed 43] epoch  15  loss 0.02660
  [seed 43] epoch  30  loss 0.02628
  [seed 43] epoch  45  loss 0.02617
  [seed 43] epoch  60  loss 0.02606
  [seed 43] epoch  75  loss 0.02603
